# SHARP — 5-Class LOEO · Fold 1 & Fold 2
### Original SHARP Architecture (Inception-ResNet CNN)

| Fold | Test Environment | Test Sessions | Train Sessions |
|------|-----------------|---------------|----------------|
| **1** | Bedroom | S1a, S1b, S1c, S2a, S2b, S3a, S4a, S4b, S5a | S6a, S6b, S7a |
| **2** | Living Room | S6a, S6b | S1a, S1b, S1c, S2a, S2b, S3a, S4a, S4b, S5a, S7a |

**Architecture**: `csi_network_inc_res` — Inception-ResNet CNN + Flatten + Dropout + Dense  
**Antenna fusion**: sum-of-logits merge (voting tiebreak) per sample window  

> Upload `doppler_traces/` as Kaggle Dataset `sharp-dataset`.  
> Enable `Settings → Accelerator → GPU T4 x2`

---
## § 1 · Imports & GPU Setup

In [ ]:
import os, json, pickle, gc, math
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix, precision_recall_fscore_support, accuracy_score
)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    print(f'[INFO] GPUs found: {gpus}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('[WARN] No GPU detected — running on CPU')
print(f'TensorFlow: {tf.__version__}')

---
## § 2 · Global Config & Fold Definitions

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DOPPLER_DIR = '/kaggle/input/doppler-traces/doppler_traces'   # adjust if needed
BASE_OUT    = '/kaggle/working'

# ── Dataset / training parameters (matching original SHARP) ───────────────
SAMPLE_LENGTH  = 340     # time dimension width  (windows_length in original)
FEATURE_LENGTH = 100     # frequency bins height  (feature_length in original)
CHANNELS       = 1       # grayscale
STRIDE_LENGTH  = 17      # stride for windowing
BATCH_SIZE     = 32
EPOCHS         = 25
LR             = 0.0001
NUM_ANTENNAS   = 4
PATIENCE       = 3       # EarlyStopping patience (original value)

# Input shapes
INPUT_NETWORK = (SAMPLE_LENGTH, FEATURE_LENGTH, CHANNELS)   # per-antenna input

# Activities — 5-class
ACTIVITIES    = ['E', 'L', 'W', 'R', 'J']   # Empty, Sitting, Walking, Running, Jumping
NUM_CLASSES   = len(ACTIVITIES)
CLASS_NAMES   = ['Empty', 'Sitting', 'Walking', 'Running', 'Jumping']
LABELS_CONSIDERED = np.arange(NUM_CLASSES)

# Activity letter → file variants (J has J1/J2 files)
_ACTIVITY_FILE_VARIANTS = {
    'E': ['E'], 'L': ['L'], 'W': ['W'], 'R': ['R'],
    'J': ['J', 'J1', 'J2'],
    'C': ['C'], 'H': ['H'], 'S': ['S'],
}

# ── Fold definitions ───────────────────────────────────────────────────────
FOLDS = [
    {
        'fold_n'         : 1,
        'test_env'       : 'Bedroom',
        'train_sessions' : ['S6a', 'S6b', 'S7a'],
        'test_sessions'  : ['S1a', 'S1b', 'S1c', 'S2a', 'S2b', 'S3a', 'S4a', 'S4b', 'S5a'],
    },
    {
        'fold_n'         : 2,
        'test_env'       : 'Living Room',
        'train_sessions' : ['S1a', 'S1b', 'S1c', 'S2a', 'S2b', 'S3a', 'S4a', 'S4b', 'S5a', 'S7a'],
        'test_sessions'  : ['S6a', 'S6b'],
    },
]

print('Config loaded.')
for f in FOLDS:
    print(f"  Fold {f['fold_n']} | Test: {f['test_env']}")
    print(f"    Train ({len(f['train_sessions'])}): {f['train_sessions']}")
    print(f"    Test  ({len(f['test_sessions'])}):  {f['test_sessions']}")

---
## § 3 · Model Architecture — `csi_network_inc_res` (Original SHARP)

`reduction_a_block_small` → 1×1 Conv → Flatten → Dropout(0.2) → Dense(logits)

In [ ]:
def conv2d_bn(x_in, filters, kernel_size, strides=(1,1), padding='same',
              activation='relu', bn=False, name=None):
    """Conv2D with optional BatchNorm and activation (from network_utility.py)."""
    x = tf.keras.layers.Conv2D(
        filters, kernel_size, strides=strides, padding=padding, name=name)(x_in)
    if bn:
        bn_name = None if name is None else name + '_bn'
        x = tf.keras.layers.BatchNormalization(axis=3, name=bn_name)(x)
    if activation is not None:
        x = tf.keras.layers.Activation(activation)(x)
    return x


def reduction_a_block_small(x_in, base_name):
    """Three-branch Inception-ResNet reduction block (from network_utility.py)."""
    # Branch 1: MaxPool 2×2
    x1 = tf.keras.layers.MaxPool2D((2,2), strides=(2,2), padding='valid')(x_in)

    # Branch 2: Conv 5×5 stride 2
    x2 = conv2d_bn(x_in, 5, (2,2), strides=(2,2), padding='valid',
                   name=base_name + 'conv2_1_res_a')

    # Branch 3: 1×1 → 3×3 → 4×4 stride 2
    x3 = conv2d_bn(x_in, 3, (1,1), name=base_name + 'conv3_1_res_a')
    x3 = conv2d_bn(x3,   6, (2,2), name=base_name + 'conv3_2_res_a')
    x3 = conv2d_bn(x3,   9, (4,4), strides=(2,2), padding='same',
                   name=base_name + 'conv3_3_res_a')

    return tf.keras.layers.Concatenate()([x1, x2, x3])


def csi_network_inc_res(input_sh, output_sh):
    """Original SHARP model (from network_utility.py / CSI_network.py)."""
    x_input = tf.keras.Input(input_sh)
    x2 = reduction_a_block_small(x_input, base_name='1st')
    x3 = conv2d_bn(x2, 3, (1,1), name='conv4')
    x  = tf.keras.layers.Flatten()(x3)
    x  = tf.keras.layers.Dropout(0.2)(x)
    x  = tf.keras.layers.Dense(output_sh, activation=None, name='dense2')(x)
    return tf.keras.Model(inputs=x_input, outputs=x, name='csi_model')


# Quick sanity check
test_model = csi_network_inc_res(INPUT_NETWORK, NUM_CLASSES)
test_model.summary(line_length=80)
del test_model
tf.keras.backend.clear_session()
print('Architecture verified.')

---
## § 4 · Dataset Utilities (adapted from `dataset_utility.py` + `CSI_doppler_create_dataset_train.py`)

In [ ]:
def load_raw_stream(filepath):
    """Load one antenna stream pickle and remove DC mean (col-wise)."""
    with open(filepath, 'rb') as fp:
        data = pickle.load(fp)            # shape: (freq_bins, time)
    data = data - np.mean(data, axis=0, keepdims=True)
    return data.T.astype(np.float32)      # → (time, freq_bins)


def load_session_activity(session_dir, session_name, act_letter, num_antennas=4):
    """
    Load all 4 antenna streams for one session+activity.
    Returns ndarray (num_antennas, freq_bins, time) or None.
    """
    variants = _ACTIVITY_FILE_VARIANTS.get(act_letter, [act_letter])
    for variant in variants:
        streams = []
        ok = True
        for ant in range(num_antennas):
            path = os.path.join(session_dir, f'{session_name}_{variant}_stream_{ant}.txt')
            if not os.path.exists(path):
                ok = False
                break
            raw = load_raw_stream(path)   # (time, freq_bins)
            streams.append(raw.T)         # → (freq_bins, time)
        if ok:
            return np.stack(streams, axis=0)   # (num_antennas, freq_bins, time)
    return None


def build_fold_data(session_list, activities, sample_length, stride_length,
                    train_ratio=0.60, val_ratio=0.20, num_antennas=4):
    """
    Build windowed train/val/test splits from raw doppler streams.
    Mirrors CSI_doppler_create_dataset_train.py logic.

    Returns:
        train_windows : list of (num_antennas, freq_bins, sample_length) arrays
        train_labels  : list of int
        val_windows, val_labels
        test_windows, test_labels
        test_windows_per_sample : used for antenna-merge fusion
        test_labels_per_sample
    """
    train_wins, train_labs = [], []
    val_wins,   val_labs   = [], []
    test_wins,  test_labs  = [], []
    # also keep per-sample (before per-antenna expand) for fusion
    test_wins_sample, test_labs_sample = [], []

    for session in session_list:
        session_dir = os.path.join(DOPPLER_DIR, session)
        if not os.path.isdir(session_dir):
            print(f'  [SKIP] {session} — directory not found')
            continue
        for act_idx, act_letter in enumerate(activities):
            data_4ant = load_session_activity(session_dir, session, act_letter, num_antennas)
            if data_4ant is None:
                continue
            # data_4ant: (num_antennas, freq_bins, time)
            time_len  = data_4ant.shape[2]
            t_end     = int(np.floor(time_len * train_ratio))
            gap       = math.ceil(sample_length / stride_length)
            v_start   = t_end + gap
            v_end     = v_start + int(np.floor(time_len * val_ratio))
            te_start  = v_end + gap

            def _slide(data_block, start_off, end_off, wins_list, labs_list,
                       per_sample_wins=None, per_sample_labs=None):
                for s in range(0, end_off - start_off - sample_length + 1, stride_length):
                    w = data_block[:, :, start_off + s: start_off + s + sample_length]
                    if per_sample_wins is not None:
                        per_sample_wins.append(w)  # (num_antennas, freq, sample_length)
                        per_sample_labs.append(act_idx)
                    # expand per antenna
                    for ant in range(num_antennas):
                        wins_list.append(w[ant])   # (freq, sample_length)
                        labs_list.append(act_idx)

            _slide(data_4ant, 0,        t_end,     train_wins, train_labs)
            _slide(data_4ant, v_start,  v_end,     val_wins,   val_labs)
            _slide(data_4ant, te_start, time_len,  test_wins,  test_labs,
                   test_wins_sample, test_labs_sample)

    print(f'  Train windows (per-ant): {len(train_wins):,}  '
          f'Val: {len(val_wins):,}  Test(per-ant): {len(test_wins):,}  '
          f'Test(per-sample): {len(test_wins_sample):,}')
    return (train_wins, train_labs, val_wins, val_labs,
            test_wins, test_labs, test_wins_sample, test_labs_sample)


def windows_to_tf_dataset(windows, labels, batch_size, shuffle=False, repeat=False):
    """
    Convert windowed data to tf.data.Dataset.
    Each window (freq_bins, sample_length) → transposed to
    (sample_length, freq_bins, 1) to match INPUT_NETWORK.
    """
    def _gen():
        for w, l in zip(windows, labels):
            # w: (freq_bins, sample_length) → (sample_length, freq_bins, 1)
            x = np.expand_dims(w.T.astype(np.float32), axis=-1)
            yield x, np.int32(l)
    ds = tf.data.Dataset.from_generator(
        _gen,
        output_signature=(
            tf.TensorSpec(shape=INPUT_NETWORK, dtype=tf.float32),
            tf.TensorSpec(shape=(),            dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(min(len(labels), 20000))
    if repeat:
        ds = ds.repeat()
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


print('Dataset utilities defined.')

---
## § 5 · Antenna Merge Fusion (original SHARP logic)

In [ ]:
def antenna_merge_fusion(test_pred_per_ant, labels_true_per_ant,
                         labels_true_per_sample, num_antennas):
    """
    Replicates the merge logic from CSI_network.py:
      - sum logits across antennas → argmax  (sum_merge)
      - majority vote with sum-merge as tiebreak  (max_merge)

    test_pred_per_ant : (N*num_antennas, num_classes) — raw logit predictions
    labels_true_per_ant : (N*num_antennas,)
    labels_true_per_sample : (N,)
    """
    n_samples    = len(labels_true_per_sample)
    pred_per_ant = np.argmax(test_pred_per_ant, axis=1)   # per-antenna class preds

    pred_max_merge = np.zeros(n_samples, dtype=np.int64)
    for i_lab in range(n_samples):
        pred_antennas = test_pred_per_ant[i_lab*num_antennas : (i_lab+1)*num_antennas, :]
        # sum-of-logits merge
        lab_merge_max = np.argmax(np.sum(pred_antennas, axis=0))
        # majority vote
        pred_max_antennas = pred_per_ant[i_lab*num_antennas : (i_lab+1)*num_antennas]
        lab_unique, count = np.unique(pred_max_antennas, return_counts=True)
        if lab_unique.shape[0] > 1:
            count_argsort    = np.flip(np.argsort(count))
            count_sort       = count[count_argsort]
            lab_unique_sort  = lab_unique[count_argsort]
            if count_sort[0] == count_sort[1] or lab_unique.shape[0] > 2:
                lab_max_merge = lab_merge_max   # tie → fall back to sum-merge
            else:
                lab_max_merge = lab_unique_sort[0]
        else:
            lab_max_merge = lab_unique[0]
        pred_max_merge[i_lab] = lab_max_merge

    return pred_max_merge


print('Antenna merge fusion defined.')

---
## § 6 · Train & Evaluate Helper — `run_fold()`

In [ ]:
def run_fold(fold_cfg):
    fold_n         = fold_cfg['fold_n']
    test_env       = fold_cfg['test_env']
    train_sess_req = fold_cfg['train_sessions']
    test_sess_req  = fold_cfg['test_sessions']
    output_dir     = os.path.join(BASE_OUT, f'fold_{fold_n}')
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'#'*70}")
    print(f'  FOLD {fold_n}  |  Test Environment: {test_env}')
    print(f"  Train ({len(train_sess_req)}): {train_sess_req}")
    print(f"  Test  ({len(test_sess_req)}):  {test_sess_req}")
    print(f"{'#'*70}")

    # Check sessions exist
    train_sessions = [s for s in train_sess_req if os.path.isdir(os.path.join(DOPPLER_DIR, s))]
    test_sessions  = [s for s in test_sess_req  if os.path.isdir(os.path.join(DOPPLER_DIR, s))]
    for m in set(train_sess_req)-set(train_sessions): print(f'  [WARN] Missing train: {m}')
    for m in set(test_sess_req) -set(test_sessions):  print(f'  [WARN] Missing test:  {m}')
    if not train_sessions or not test_sessions:
        print('[ERROR] Cannot run — critical sessions missing!'); return None

    # ── Build windowed datasets ─────────────────────────────────────────────
    print(f'\n[Fold {fold_n}] Building TRAIN windows ...')
    (train_wins, train_labs,
     val_wins,   val_labs,
     _,          _,
     _,          _) = build_fold_data(
        train_sessions, ACTIVITIES, SAMPLE_LENGTH, STRIDE_LENGTH,
        train_ratio=0.60, val_ratio=0.20, num_antennas=NUM_ANTENNAS)

    print(f'\n[Fold {fold_n}] Building TEST windows ...')
    (_,           _,
     _,           _,
     test_wins,  test_labs,
     test_wins_sample, test_labs_sample) = build_fold_data(
        test_sessions, ACTIVITIES, SAMPLE_LENGTH, STRIDE_LENGTH,
        train_ratio=0.0, val_ratio=0.0, num_antennas=NUM_ANTENNAS)

    # ── tf.data pipelines ───────────────────────────────────────────────────
    print(f'\n[Fold {fold_n}] Building tf.data pipelines ...')
    n_train    = len(train_labs)
    n_val      = len(val_labs)
    n_test_ant = len(test_labs)
    ds_train = windows_to_tf_dataset(train_wins, train_labs, BATCH_SIZE, shuffle=True,  repeat=True)
    ds_val   = windows_to_tf_dataset(val_wins,   val_labs,   BATCH_SIZE, shuffle=False, repeat=True)
    ds_test  = windows_to_tf_dataset(test_wins,  test_labs,  BATCH_SIZE, shuffle=False, repeat=False)
    steps_tr = max(1, int(np.ceil(n_train    / BATCH_SIZE)))
    steps_va = max(1, int(np.ceil(n_val      / BATCH_SIZE)))
    steps_te = max(1, int(np.ceil(n_test_ant / BATCH_SIZE)))

    # ── Build model ─────────────────────────────────────────────────────────
    print(f'\n[Fold {fold_n}] Building csi_network_inc_res model ...')
    tf.keras.backend.clear_session()
    model = csi_network_inc_res(INPUT_NETWORK, NUM_CLASSES)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
    loss_fn   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer=optimizer, loss=loss_fn,
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])
    model.summary(line_length=80)

    # ── Callbacks ───────────────────────────────────────────────────────────
    model_path = os.path.join(output_dir, f'fold{fold_n}_E,L,W,R,J_network.h5')
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            model_path, save_freq='epoch', save_best_only=True,
            monitor='val_sparse_categorical_accuracy', verbose=1),
    ]

    # ── Train ────────────────────────────────────────────────────────────────
    print(f'\n[Fold {fold_n}] Training (up to {EPOCHS} epochs, patience={PATIENCE}) ...')
    history = model.fit(
        ds_train, epochs=EPOCHS, steps_per_epoch=steps_tr,
        validation_data=ds_val, validation_steps=steps_va,
        callbacks=callbacks, verbose=1)
    with open(os.path.join(output_dir, 'history.json'), 'w') as fp:
        json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, fp, indent=2)

    # ── Load best model ──────────────────────────────────────────────────────
    model = tf.keras.models.load_model(model_path)

    # ── Predict (per-antenna, all test windows) ──────────────────────────────
    print(f'\n[Fold {fold_n}] Predicting on {n_test_ant:,} test windows ...')
    test_pred_list = model.predict(ds_test, steps=steps_te)[:n_test_ant]
    test_pred_ant  = np.argmax(test_pred_list, axis=1)
    labels_true_ant = np.array(test_labs)

    # ── Single-antenna metrics ───────────────────────────────────────────────
    conf_single = confusion_matrix(labels_true_ant, test_pred_ant, labels=LABELS_CONSIDERED)
    prec_s, rec_s, f1_s, _ = precision_recall_fscore_support(
        labels_true_ant, test_pred_ant, labels=LABELS_CONSIDERED, zero_division=0)
    acc_s = accuracy_score(labels_true_ant, test_pred_ant)

    # ── Antenna-merge metrics ────────────────────────────────────────────────
    labels_true_sample = np.array(test_labs_sample)
    pred_merged = antenna_merge_fusion(
        test_pred_list, labels_true_ant, labels_true_sample, NUM_ANTENNAS)
    conf_merge = confusion_matrix(labels_true_sample, pred_merged, labels=LABELS_CONSIDERED)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        labels_true_sample, pred_merged, labels=LABELS_CONSIDERED, zero_division=0)
    acc_m = accuracy_score(labels_true_sample, pred_merged)

    # ── Print results ────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f'  Fold {fold_n} — Test: {test_env}')
    print(f"{'='*70}")
    print(f'  [Single-antenna]')
    print(f'    Accuracy      : {acc_s:.4f} ({acc_s*100:.2f}%)')
    print(f'    Macro Precision: {np.mean(prec_s):.4f}')
    print(f'    Macro Recall   : {np.mean(rec_s):.4f}')
    print(f'    Macro F1       : {np.mean(f1_s):.4f}')
    print(f"  {'─'*68}")
    print(f'  [4-Antenna Merge (sum-logits + voting tiebreak)]')
    print(f'    Accuracy      : {acc_m:.4f} ({acc_m*100:.2f}%)')
    print(f'    Macro Precision: {np.mean(prec_m):.4f}')
    print(f'    Macro Recall   : {np.mean(rec_m):.4f}')
    print(f'    Macro F1       : {np.mean(f1_m):.4f}')
    print(f"  {'─'*68}")
    for i, cls in enumerate(CLASS_NAMES):
        print(f'  {cls:<20} [merge] P={prec_m[i]:.4f}  R={rec_m[i]:.4f}  F1={f1_m[i]:.4f}')
    print(f"  {'─'*68}")
    print(f'\n  Confusion Matrix — Merged (rows=true, cols=pred):')
    print(f'  Labels: {CLASS_NAMES}')
    print(conf_merge)
    print(f"{'='*70}")

    # ── Save results ─────────────────────────────────────────────────────────
    results = {
        'fold': fold_n, 'test_env': test_env,
        # single-antenna
        'accuracy_single'   : float(acc_s),
        'macro_prec_single' : float(np.mean(prec_s)),
        'macro_rec_single'  : float(np.mean(rec_s)),
        'macro_f1_single'   : float(np.mean(f1_s)),
        'precision_single'  : prec_s.tolist(), 'recall_single': rec_s.tolist(), 'fscore_single': f1_s.tolist(),
        'conf_matrix_single': conf_single.tolist(),
        # merged
        'accuracy_merge'    : float(acc_m),
        'macro_prec_merge'  : float(np.mean(prec_m)),
        'macro_rec_merge'   : float(np.mean(rec_m)),
        'macro_f1_merge'    : float(np.mean(f1_m)),
        'precision_merge'   : prec_m.tolist(), 'recall_merge': rec_m.tolist(), 'fscore_merge': f1_m.tolist(),
        'conf_matrix_merge' : conf_merge.tolist(),
        'classes'           : CLASS_NAMES,
        'n_train': n_train, 'n_val': n_val, 'n_test_ant': n_test_ant,
        'n_test_sample': len(test_labs_sample),
    }
    out_file = os.path.join(output_dir, 'results.json')
    with open(out_file, 'w') as fp:
        json.dump(results, fp, indent=2)
    print(f'  Results saved → {out_file}')

    # ── Free memory ──────────────────────────────────────────────────────────
    del model, train_wins, val_wins, test_wins, test_wins_sample
    del ds_train, ds_val, ds_test
    gc.collect()
    tf.keras.backend.clear_session()

    return results


print('run_fold() defined.')

---
## § 7 · Run Fold 1 — Test: Bedroom

> Train: Living Room (S6a, S6b) + Laboratory (S7a) → 3 sessions
> Test:  Bedroom (S1a–S5a) → 9 sessions

In [ ]:
results_fold1 = run_fold(FOLDS[0])

---
## § 8 · Run Fold 2 — Test: Living Room

> Train: Bedroom (S1a–S5a) + Laboratory (S7a) → 10 sessions
> Test:  Living Room (S6a, S6b) → 2 sessions

In [ ]:
results_fold2 = run_fold(FOLDS[1])

---
## § 9 · Cross-Fold Summary

In [ ]:
all_results = [r for r in [results_fold1, results_fold2] if r is not None]

if all_results:
    print(f"\n{'=':=<70}")
    print('  SHARP LOEO — Cross-Fold Summary  (Fold 1 + Fold 2)')
    print(f"{'=':=<70}")
    print(f"  {'Fold':<6} {'Test Env':<16} {'Acc(single)':>12} {'Acc(merge)':>11} {'F1(merge)':>10}")
    print(f"  {'-':─<60}")
    for r in all_results:
        print(f"  Fold {r['fold']}  {r['test_env']:<16}  "
              f"{r['accuracy_single']*100:>8.2f}%    "
              f"{r['accuracy_merge']*100:>7.2f}%   "
              f"{r['macro_f1_merge']:>9.4f}")
    print(f"  {'-':─<60}")
    mean_acc_s = np.mean([r['accuracy_single'] for r in all_results])
    mean_acc_m = np.mean([r['accuracy_merge']  for r in all_results])
    mean_f1_m  = np.mean([r['macro_f1_merge']  for r in all_results])
    print(f"  Mean            {'':16}  {mean_acc_s*100:>8.2f}%    {mean_acc_m*100:>7.2f}%   {mean_f1_m:>9.4f}")
    print(f"{'=':=<70}")

    print(f"\n  Per-Class Merged F1")
    hdr = f"  {'Class':<20}" + ''.join([f'  Fold{r["fold"]}' for r in all_results]) + '   Mean'
    print(hdr)
    print(f"  {'-':─<55}")
    for i, cls in enumerate(CLASS_NAMES):
        f1s = [r['fscore_merge'][i] for r in all_results]
        print(f"  {cls:<20}" + ''.join([f'  {v:.4f}' for v in f1s]) + f'  {np.mean(f1s):.4f}')

    summary = {
        'folds'           : all_results,
        'mean_acc_single' : float(mean_acc_s),
        'mean_acc_merge'  : float(mean_acc_m),
        'mean_f1_merge'   : float(mean_f1_m),
    }
    summ_path = os.path.join(BASE_OUT, 'loeo_fold1_fold2_summary.json')
    with open(summ_path, 'w') as fp:
        json.dump(summary, fp, indent=2)
    print(f'\n  Combined summary → {summ_path}')
else:
    print('[WARN] No fold results to summarise.')